# HalfCheetah RL Competition — Colab Starter

Train and submit directly from the notebook. **Edit the config dict below** to change algorithm, phase, or hyperparameters, then run the training cell.
- **Phase 1:** `HalfCheetah-v5:run` — maximize forward velocity  
- **Phase 2:** `HalfCheetah-v5:backflip` — maximize angular velocity  
- **Phase 3:** `HalfCheetah-v5:efficient` — velocity per unit energy

In [ ]:
!pip install -q gymnasium[mujoco] stable-baselines3 pyyaml moviepy wandb

## 0. Clone the starter kit

This clones the student repo so you have access to `src/train.py`, `src/eval_submit.py`, and the eval utilities.

In [ ]:
import os

REPO_DIR = "/content/aidl-rl-students"

if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/juanjo3ns/aidl-rl-students.git {REPO_DIR}
    print(f"Cloned to {REPO_DIR}")
else:
    print(f"Repo already present at {REPO_DIR}")

import sys
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

## W&B login

In [ ]:
import wandb
wandb.login()

## Phase 1 — Run

Edit the **config** dict below: change `algorithm` (ppo, sac, td3, a2c) or any hyperparameters. Train, then submit to the leaderboard.

In [ ]:
import sys
import os

REPO_DIR = "/content/aidl-rl-students"  # set by the clone cell above

# --- Edit this config: phase, algorithm, and hyperparameters ---
config = {
    "algorithm": "sac",           # one of: ppo, sac, td3, a2c
    "policy": "MlpPolicy",
    "env": {
        "id": "HalfCheetah-v5:run",   # or :backflip, :efficient
    },
    "hyperparameters": {
        "learning_rate": 3.0e-4,
        "buffer_size": 300_000,
        "batch_size": 256,
        "tau": 0.005,
        "gamma": 0.99,
        "train_freq": 1,
        "gradient_steps": 1,
        "ent_coef": "auto",
        "target_update_interval": 1,
    },
    "training": {
        "total_steps": 500_000,   # reduce for quick tests (e.g. 50_000)
        "seed": 42,
        "n_envs": 1,
        "verbose": 0,
        "model_path": "models/sac_agent.zip",
        "log_interval": 1000,
    },
    "evaluation": {
        "interval": 25_000,
        "episodes": 5,
        "video_interval": 100_000,
        "video_fps": 10,
    },
    "wandb": {"project": "aidl-rl-benchmark", "tags": ["training", "sac"]},
}

# PPO example (uncomment and set config = ppo_config to use):
# ppo_config = {**config, "algorithm": "ppo", "hyperparameters": {"learning_rate": 3e-4, "n_steps": 2048, "batch_size": 64, "n_epochs": 10, "gamma": 0.99, "gae_lambda": 0.95, "clip_range": 0.2, "ent_coef": 0.01, "vf_coef": 0.5, "max_grad_norm": 0.5}, "training": {**config["training"], "model_path": "models/ppo_agent.zip"}}

In [ ]:
from src.train import run_training

# Train (set use_wandb=True after wandb.login() in the optional cell above)
model_path = run_training(config, use_wandb=True)
print("Model saved at:", model_path)

### Submit to the leaderboard

Set your session code, team name, and API URL, then run. Use the same `env_id` and `algo` as in your config.

In [ ]:
from src.eval_submit import evaluate_and_submit

# Edit these:
SESSION_CODE = "YOUR_SESSION_CODE"
TEAM_NAME = "Team Turbo"
API_URL = "https://YOUR-DOMAIN/api/submit"

# Model path and algo must match your config above
response = evaluate_and_submit(
    SESSION_CODE,
    TEAM_NAME,
    config["env"]["id"],
    config["algorithm"],
    str(model_path),
    API_URL,
    seeds_path=os.path.join(REPO_DIR, "eval", "seeds.json"),
    episodes_per_seed=5,
    use_wandb=True,
    eval_video=True,
    video_format="mp4",
)

## Phase 2 — Backflip (Exercise)

For Phase 2 (`HalfCheetah-v5:backflip`), you need to implement a custom reward that encourages the cheetah to do backflips.

The `BackflipReward` class wraps the environment and replaces the default reward. The skeleton is already in `src/train.py` and `eval/run_eval.py` — **fill in the reward computation** in the cell below, then copy your solution into both files.

**What you know:**
- `self.unwrapped.data.qvel` gives you the joint velocity vector (generalized velocities)
- Indices `3:6` of `qvel` correspond to the **torso angular velocity** `(wx, wy, wz)`
- Your reward should be the **L2 norm** of that angular velocity vector

**What you need to do:** Replace the `new_reward = 0.0` line with your implementation (~3 lines).

In [ ]:
import gymnasium as gym
import numpy as np

class BackflipReward(gym.RewardWrapper):
    """Reward = L2 norm of torso angular velocity."""

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        # ── TODO: compute the backflip reward ────────────────────────
        # Hints:
        #   - Access joint velocities via: self.unwrapped.data.qvel
        #   - Indices 3:6 are the torso angular velocity (wx, wy, wz)
        #   - The reward should be the L2 norm of that angular velocity vector
        #   - Handle the edge case where qvel has fewer than 6 elements
        new_reward = 0.0  # ← replace this with your implementation
        # ─────────────────────────────────────────────────────────────
        return obs, new_reward, terminated, truncated, info

# --- Quick sanity check ---
env = gym.make("HalfCheetah-v5")
wrapped = BackflipReward(env)
obs, _ = wrapped.reset(seed=42)
obs, rew, *_ = wrapped.step(wrapped.action_space.sample())
print(f"Backflip reward for a random action: {rew:.4f}")
assert rew >= 0, "Reward should be non-negative (it's an L2 norm!)"
if rew == 0.0:
    print("⚠️  Reward is 0.0 — did you forget to implement it?")
else:
    print("✅ Looks good! Now copy your implementation into src/train.py and eval/run_eval.py")
wrapped.close()

### Train and submit Phase 2

Once your `BackflipReward` passes the sanity check above:

1. **Copy** your implementation into `src/train.py` and `eval/run_eval.py` (replace the TODO blocks in `BackflipReward.step`)
2. **Change** `config["env"]["id"]` to `"HalfCheetah-v5:backflip"` in the config cell above
3. **Re-run** the training and submit cells (Phase 1 section) to train and submit your backflip agent

## Tips
- **Phase 3:** set `config["env"]["id"]` to `HalfCheetah-v5:efficient` (the `EfficientRunReward` wrapper is already implemented for you).
- **Try PPO or TD3:** set `config["algorithm"]` to `"ppo"` or `"td3"` and adjust `hyperparameters` (see `src/configs/*.yaml` in the repo for full examples).
- **Quick test:** set `config["training"]["total_steps"]` to e.g. `50_000` and re-run training.